# SMILESGNN Inference System
## From Virtual Screening to Lead Optimization

This notebook implements a two-stage inference system for clinical toxicity prediction:

| Stage | Task | Input | Speed | Output |
|---|---|---|---|---|
| **A** | Prediction | SMILES list | Fast (~ms/mol) | P(toxic), ranked table |
| **B** | Interpretation | Single SMILES | Slow (~s/mol) | Atom/bond heatmap |

**Recommended workflow** (mirrors real drug discovery practice):
```
Compound library → [Stage A] Batch predict all → Rank by P(toxic)
                                                        │
                                    Flagged toxic compounds only
                                                        │
                                  → [Stage B] Single explain each hit
                                  → Atom heatmap → structural modification
```

---
> **Prerequisite**: `models/smilesgnn_model/best_model.pt` must exist.  
> Run `python scripts/train_hybrid.py --device cuda` if not present.

## 0. Setup

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from torch_geometric.data import Batch
from torch.utils.data import DataLoader

from src.data import load_clintox
from src.graph_data import smiles_list_to_pyg_dataset, smiles_to_pyg_data, get_feature_dims
from src.graph_models_hybrid import create_hybrid_model
from src.gnn_explainer import explain_molecule, visualize_explanation

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_DIR   = PROJECT_ROOT / 'models' / 'smilesgnn_model'
CONFIG_PATH = PROJECT_ROOT / 'config' / 'smilesgnn_config.yaml'

print(f'Device     : {DEVICE}')
print(f'Model dir  : {MODEL_DIR}')
print(f'Checkpoint : {MODEL_DIR / "best_model.pt"}')

## 1. Load Model

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)
mc = config['model']

# ── Tokenizer first (actual vocab size governs embedding shape) ─────────────
with open(MODEL_DIR / 'tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
actual_vocab_size = len(tokenizer.token_to_id)

# ── Rebuild model ───────────────────────────────────────────────────────────
num_node_features, num_edge_features = get_feature_dims()
model = create_hybrid_model(
    num_node_features = num_node_features,
    num_edge_features = num_edge_features,
    hidden_dim        = int(mc['hidden_dim']),
    num_graph_layers  = int(mc['num_graph_layers']),
    graph_model       = mc['graph_model'],
    num_heads         = int(mc['num_heads']),
    dropout           = float(mc['dropout']),
    use_residual      = bool(mc.get('use_residual', True)),
    use_jk            = bool(mc.get('use_jk', True)),
    jk_mode           = mc.get('jk_mode', 'cat'),
    graph_pooling     = mc.get('graph_pooling', 'meanmax'),
    smiles_vocab_size = actual_vocab_size,
    smiles_d_model    = int(mc['smiles_d_model']),
    smiles_num_layers = int(mc['smiles_num_layers']),
    fusion_method     = mc.get('fusion_method', 'attention'),
)
state = torch.load(MODEL_DIR / 'best_model.pt', map_location=DEVICE)
model.load_state_dict(state)
model.to(DEVICE)
model.eval()

# ── HybridModelWrapper ───────────────────────────────────────────────────────
# Routes SMILES token IDs stored in the batch object to model.forward().
# Must match the wrapper used during training.
class HybridModelWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, batch):
        return self.model(
            batch,
            smiles_token_ids      = batch.smiles_token_ids       if hasattr(batch, 'smiles_token_ids')       else None,
            smiles_attention_mask = batch.smiles_attention_masks if hasattr(batch, 'smiles_attention_masks') else None,
        )

wrapped_model = HybridModelWrapper(model)
wrapped_model.eval()

print(f'Model loaded  — {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'Vocabulary    — {actual_vocab_size} tokens')
print(f'Node features — {num_node_features} | Edge features — {num_edge_features}')

## 2. Batch Prediction Engine

Defines `predict_batch()` — the fast screening function.  
Processes a list of SMILES and returns a ranked DataFrame with toxicity probabilities.

> Compounds that RDKit cannot parse (e.g. organometallics, exotic coordination chemistry)  
> are silently marked as `'Parse error'` and excluded from model scoring.

In [ ]:
class _HybridDataset:
    """Attaches SMILES token IDs to each PyG Data object for batched inference."""
    def __init__(self, pyg_dataset, smiles_list, tok):
        self.pyg_dataset = pyg_dataset
        self.smiles_list = smiles_list
        self.tok = tok
    def __len__(self): return len(self.pyg_dataset)
    def __getitem__(self, idx):
        data = self.pyg_dataset[idx]
        ids, mask = self.tok.encode(self.smiles_list[idx])
        data.smiles_token_ids      = torch.tensor(ids,  dtype=torch.long)
        data.smiles_attention_mask = torch.tensor(mask, dtype=torch.long)
        return data

def _collate(batch):
    b = Batch.from_data_list(batch)
    if hasattr(batch[0], 'smiles_token_ids'):
        b.smiles_token_ids       = torch.stack([x.smiles_token_ids      for x in batch])
        b.smiles_attention_masks = torch.stack([x.smiles_attention_mask for x in batch])
    return b


def predict_batch(
    smiles_list : list,
    names       : list  = None,
    true_labels : list  = None,
    threshold   : float = 0.5,
    batch_size  : int   = 32,
) -> pd.DataFrame:
    """
    Stage A — batch toxicity prediction.

    Parameters
    ----------
    smiles_list  : list of SMILES strings
    names        : optional compound names (auto-generated if None)
    true_labels  : optional ground-truth labels (0/1)
    threshold    : decision boundary (default 0.5)
    batch_size   : GPU mini-batch size

    Returns
    -------
    pd.DataFrame sorted by P(toxic) descending.
    Compounds that fail RDKit featurisation appear at the bottom as 'Parse error'.
    """
    valid, invalid = [], []

    for i, smi in enumerate(smiles_list):
        name = names[i] if names else f'Mol-{i:03d}'
        lbl  = true_labels[i] if true_labels is not None else None
        try:
            # smiles_to_pyg_data raises RuntimeError for unusual chemistries
            # (e.g. organometallics) where RDKit cannot compute implicit valence
            d = smiles_to_pyg_data(smi, label=lbl if lbl is not None else 0)
        except Exception:
            d = None
        if d is None:
            invalid.append({
                'Name': name, 'SMILES': smi, 'P(toxic)': None,
                'Predicted': 'Parse error',
                'True label': ('Toxic' if lbl == 1 else 'Non-toxic') if lbl is not None else '—',
                'Correct': '—',
            })
        else:
            valid.append((i, name, smi, lbl, d))

    if not valid:
        print(f'WARNING: all {len(smiles_list)} compounds failed RDKit parsing.')
        return pd.DataFrame(invalid)

    _, vnames, vsmiles, vlabels, pyg_list = zip(*valid)
    dataset = _HybridDataset(list(pyg_list), list(vsmiles), tokenizer)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=_collate)

    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch  = batch.to(DEVICE)
            logits = wrapped_model(batch).squeeze(-1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist() if probs.ndim > 0 else [float(probs)])

    rows = []
    for name, smi, lbl, prob in zip(vnames, vsmiles, vlabels, all_probs):
        pred    = 1 if prob >= threshold else 0
        correct = (pred == lbl) if lbl is not None else None
        rows.append({
            'Name'      : name,
            'SMILES'    : smi,
            'P(toxic)'  : round(prob, 4),
            'Predicted' : 'Toxic' if pred == 1 else 'Non-toxic',
            'True label': ('Toxic' if lbl == 1 else 'Non-toxic') if lbl is not None else '—',
            'Correct'   : ('✓' if correct else '✗') if correct is not None else '—',
        })

    rows.extend(invalid)
    df = pd.DataFrame(rows).sort_values('P(toxic)', ascending=False, na_position='last')
    return df.reset_index(drop=True)


print('predict_batch() ready.')

---
# Workflow A — Virtual Screening

> *"Which of these compounds should I deprioritize before synthesis?"*

We simulate two compound libraries:
1. **ClinTox test set** (148 molecules) — ground truth available for validation
2. **Curated reference panel** (8 well-known drugs / toxins) — external sanity check

### A1. Load compound library

In [ ]:
dc = config['data']
_, _, test_df = load_clintox(
    cache_dir  = str(PROJECT_ROOT / dc['cache_dir']),
    split_type = dc['split_type'],
    seed       = dc['seed'],
)

print(f'ClinTox test set: {len(test_df)} compounds')
print(f'  Toxic    : {test_df["CT_TOX"].sum()} ({test_df["CT_TOX"].mean()*100:.1f}%)')
print(f'  Non-toxic: {(test_df["CT_TOX"]==0).sum()} ({(1-test_df["CT_TOX"].mean())*100:.1f}%)')

### A2. Run batch prediction — full library

In [ ]:
results_df = predict_batch(
    smiles_list = test_df['smiles'].tolist(),
    true_labels = test_df['CT_TOX'].tolist(),
    threshold   = 0.5,
    batch_size  = 32,
)

n_predicted_toxic    = (results_df['Predicted'] == 'Toxic').sum()
n_predicted_nontoxic = (results_df['Predicted'] == 'Non-toxic').sum()

print(f'Screened : {len(results_df)} compounds')
print(f'  Flagged toxic    : {n_predicted_toxic}')
print(f'  Passed (non-toxic): {n_predicted_nontoxic}')
print()
print('Top 10 compounds by P(toxic):')
display(results_df.head(10)[['Name', 'SMILES', 'P(toxic)', 'Predicted', 'True label', 'Correct']])

### A3. Visualize predicted toxicity distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: probability histogram ─────────────────────────────────────────────
ax = axes[0]
toxic_probs    = results_df[results_df['True label'] == 'Toxic']['P(toxic)'].dropna()
nontoxic_probs = results_df[results_df['True label'] == 'Non-toxic']['P(toxic)'].dropna()
ax.hist(nontoxic_probs, bins=25, color='steelblue', alpha=0.7, label='Non-toxic (true)', edgecolor='white')
ax.hist(toxic_probs,    bins=25, color='salmon',    alpha=0.9, label='Toxic (true)',     edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.5')
ax.set_xlabel('P(toxic)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Predicted toxicity probability\ndistribution (ClinTox test set)', fontsize=12)
ax.legend(fontsize=10)

# ── Right: decision pie chart ────────────────────────────────────────────────
ax2 = axes[1]
sizes  = [n_predicted_toxic, n_predicted_nontoxic]
labels = [f'Flagged toxic\n({n_predicted_toxic})', f'Passed\n({n_predicted_nontoxic})']
colors = ['salmon', 'steelblue']
ax2.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 11})
ax2.set_title('Screening decision (threshold = 0.5)', fontsize=12)

plt.tight_layout()
plt.show()

print(f'Separation: {len(toxic_probs[toxic_probs>=0.5])}/{len(toxic_probs)} '
      f'true toxic flagged  |  '
      f'{len(nontoxic_probs[nontoxic_probs<0.5])}/{len(nontoxic_probs)} '
      f'true non-toxic passed')

### A4. External sanity check — curated reference panel

Eight well-characterized compounds covering diverse toxicity mechanisms.  
All SMILES are standard organic structures compatible with RDKit featurisation.

> **Note on organometallics**: compounds such as Cisplatin (`[NH3][Pt](Cl)(Cl)[NH3]`)  
> use coordination chemistry that RDKit cannot fully sanitize — implicit valence for Pt  
> is undefined. Such compounds are reported as `'Parse error'` and excluded from scoring.  
> The model is trained on organic SMILES only and is not designed for metal complexes.

In [ ]:
reference_panel = [
    # (name,              SMILES,                                                              label, notes)
    ('Thalidomide',       'O=C1CCC(=O)N1c1ccc2c(c1)C(=O)N(C2=O)',                            1, 'Teratogen — glutarimide scaffold, CRBN binder'),
    ('5-Fluorouracil',    'O=c1[nH]cc(F)c(=O)[nH]1',                                         1, 'Antimetabolite — myelosuppression, GI toxicity'),
    ('Methotrexate',      'CN(Cc1cnc2nc(N)nc(N)c2n1)c1ccc(cc1)C(=O)NC(CCC(O)=O)C(O)=O',    1, 'Antifolate — hepatotoxic, renal toxicity'),
    ('Benzo[a]pyrene',    'c1ccc2c(c1)ccc1ccc3cccc4ccc2c1c34',                               1, 'Carcinogen — PAH, DNA adduct formation'),
    ('Aspirin',           'CC(=O)Oc1ccccc1C(=O)O',                                           0, 'Safe NSAID — low clinical toxicity'),
    ('Caffeine',          'Cn1cnc2c1c(=O)n(C)c(=O)n2C',                                      0, 'Methylxanthine — safe stimulant'),
    ('Warfarin',          'CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O',                          0, 'Anticoagulant — narrow window, not inherently toxic'),
    ('Acetaminophen',     'CC(=O)Nc1ccc(O)cc1',                                              0, 'Safe analgesic — toxic only at overdose'),
]

ref_names  = [r[0] for r in reference_panel]
ref_smiles = [r[1] for r in reference_panel]
ref_labels = [r[2] for r in reference_panel]
ref_notes  = [r[3] for r in reference_panel]

ref_df = predict_batch(
    smiles_list = ref_smiles,
    names       = ref_names,
    true_labels = ref_labels,
    threshold   = 0.5,
)
ref_df['Notes'] = ref_notes

print('Reference panel predictions:')
print()
display(ref_df[['Name', 'P(toxic)', 'Predicted', 'True label', 'Correct', 'Notes']])

In [ ]:
# Bar chart: P(toxic) for reference panel
valid_ref = ref_df[ref_df['P(toxic)'].notna()]
probs_ref = valid_ref['P(toxic)'].tolist()
names_ref = valid_ref['Name'].tolist()
colors_bar = ['salmon' if p >= 0.5 else 'steelblue' for p in probs_ref]

fig, ax = plt.subplots(figsize=(11, max(4, len(names_ref) * 0.55)))
bars = ax.barh(names_ref[::-1], probs_ref[::-1], color=colors_bar[::-1],
               edgecolor='black', linewidth=0.5, height=0.6)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Decision threshold')
ax.set_xlabel('P(toxic)', fontsize=12)
ax.set_title('SMILESGNN predictions — Reference Panel', fontsize=13)
ax.set_xlim(0, 1.08)
ax.legend(fontsize=10)
for bar, prob in zip(bars[::-1], probs_ref[::-1]):
    ax.text(min(prob + 0.02, 1.0), bar.get_y() + bar.get_height() / 2,
            f'{prob:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
# Workflow B — Lead Optimization: Single-Molecule Deep Dive

> *"This compound was flagged. Which specific atoms drive the prediction?  
> What structural modification might reduce toxicity?"*

We use **GNNExplainer** to generate atom- and bond-level importance scores for one compound.  
The SMILES (Transformer) embedding is frozen; only the GATv2 graph pathway is perturbed,  
giving explanations faithful to the model's structural representation.

### B1. Select a compound to investigate

Change `TARGET_SMILES` to any flagged compound from Workflow A.

In [ ]:
# ── Choose your compound ─────────────────────────────────────────────────────
TARGET_NAME   = 'Thalidomide'
TARGET_SMILES = 'O=C1CCC(=O)N1c1ccc2c(c1)C(=O)N(C2=O)'

# Stage A: quick prediction
quick = predict_batch([TARGET_SMILES], names=[TARGET_NAME])
prob  = quick.iloc[0]['P(toxic)']
pred  = quick.iloc[0]['Predicted']

print(f'Compound  : {TARGET_NAME}')
print(f'SMILES    : {TARGET_SMILES}')
print(f'P(toxic)  : {prob:.4f}')
print(f'Prediction: {pred}')
print()

explain_target = 1 if prob >= 0.5 else 0
if explain_target == 0:
    print('NOTE: predicted non-toxic — Stage B will explain the non-toxic prediction.')
else:
    print('Proceeding to structural explanation (Stage B) ...')

### B2. Run GNNExplainer

In [ ]:
from rdkit import Chem

pyg_data = smiles_to_pyg_data(TARGET_SMILES, label=explain_target)

explanation = explain_molecule(
    smiles       = TARGET_SMILES,
    model        = model,       # underlying model (not the wrapper)
    tokenizer    = tokenizer,
    pyg_data     = pyg_data,
    device       = DEVICE,
    epochs       = 300,         # higher = more stable masks
    target_class = explain_target,
)

print(f'GNNExplainer complete.')
print(f'  Atoms : {len(explanation["atom_importance"])}')
print(f'  Bonds : {len(explanation["bond_importance"])}')

### B3. Atom & bond importance heatmap

In [ ]:
visualize_explanation(
    explanation,
    figsize        = (13, 5),
    atom_threshold = 0.4,
    bond_threshold = 0.4,
    # save_path    = f'explanation_{TARGET_NAME}.png'   # uncomment to save
)

### B4. Ranked atom importance table

In [ ]:
mol      = Chem.MolFromSmiles(TARGET_SMILES)
atom_imp = explanation['atom_importance']
bond_imp = explanation['bond_importance']

# ── Atom table ───────────────────────────────────────────────────────────────
atom_rows = []
for atom in mol.GetAtoms():
    idx      = atom.GetIdx()
    imp      = float(atom_imp[idx])
    atom_rows.append({
        'Atom idx'     : idx,
        'Element'      : atom.GetSymbol(),
        'Importance'   : round(imp, 4),
        'Signal'       : '█' * int(imp * 10),
        'Hybridization': str(atom.GetHybridization()).split('.')[-1],
        'In ring'      : '✓' if atom.IsInRing()       else '',
        'Aromatic'     : '✓' if atom.GetIsAromatic()  else '',
    })

atom_df = pd.DataFrame(atom_rows).sort_values('Importance', ascending=False)
print(f'Atom importance — {TARGET_NAME}')
print(f'P(toxic) = {explanation["prediction_prob"]:.4f}  |  '
      f'Predicted: {"Toxic" if explanation["predicted_class"]==1 else "Non-toxic"}')
print()
display(atom_df.reset_index(drop=True))

# ── Bond table (top 10) ──────────────────────────────────────────────────────
bond_rows = []
for bond in mol.GetBonds():
    k   = bond.GetIdx()
    imp = float(bond_imp[k]) if k < len(bond_imp) else 0.0
    a1  = mol.GetAtomWithIdx(bond.GetBeginAtomIdx()).GetSymbol()
    a2  = mol.GetAtomWithIdx(bond.GetEndAtomIdx()).GetSymbol()
    bond_rows.append({
        'Bond idx'  : k,
        'Atoms'     : f'{a1}({bond.GetBeginAtomIdx()})–{a2}({bond.GetEndAtomIdx()})',
        'Type'      : str(bond.GetBondTypeAsDouble()),
        'Importance': round(imp, 4),
    })

bond_df = pd.DataFrame(bond_rows).sort_values('Importance', ascending=False).head(10)
print()
print('Top-10 bond importances:')
display(bond_df.reset_index(drop=True))

### B5. Structural interpretation guide

**For Thalidomide:**
- High-importance atoms are expected to cluster around the **glutarimide ring**  
  (atoms: N–C=O–CH₂–CH₂–C=O). This is the teratogenic pharmacophore.  
  The imide NH undergoes base-catalysed hydrolysis → reactive species → binds cereblon (CRBN)
- The **phthalimide ring** contributes sedative/CNS activity; high importance there  
  may reflect hERG or CNS signal capture

**General modification strategy:**

| Step | Action |
|---|---|
| 1 | Identify top-3 atoms by importance |
| 2 | Determine their functional group context (imide? amine? halogen?) |
| 3 | Propose bioisosteric replacement (e.g. imide → amide reduces reactivity) |
| 4 | Run `predict_batch([modified_smiles])` → confirm P(toxic) drops |
| 5 | If P drops significantly: candidate for synthesis and wet-lab validation |

---
# Integrated Pipeline — Batch Predict → Single Explain on Hits

> *The most realistic drug discovery workflow.*  
> Screen the full library (Stage A), then deep-dive into the top flagged hits (Stage B).

We take the **top 3 predicted-toxic compounds** from the ClinTox test set  
and produce a structural explanation for each.

In [ ]:
# ── Step 1: top-3 toxic predictions from test set ────────────────────────────
toxic_hits = results_df[results_df['Predicted'] == 'Toxic'].head(3)

print('Top-3 predicted-toxic compounds selected for deep analysis:')
print()
for _, row in toxic_hits.iterrows():
    print(f"  {row['Name']}  |  P(toxic) = {row['P(toxic)']:.4f}  |  True: {row['True label']}")

In [ ]:
# ── Step 2: explain each hit ──────────────────────────────────────────────────
pipeline_results = []

for n, (_, row) in enumerate(toxic_hits.iterrows()):
    smi  = row['SMILES']
    name = row['Name']
    prob = row['P(toxic)']
    print(f'[{n+1}/3] {name}  P={prob:.3f} …')

    try:
        pyg = smiles_to_pyg_data(smi, label=1)
    except Exception:
        pyg = None
    if pyg is None:
        print(f'  Skipping — RDKit parse error')
        continue

    res = explain_molecule(
        smiles       = smi,
        model        = model,
        tokenizer    = tokenizer,
        pyg_data     = pyg,
        device       = DEVICE,
        epochs       = 300,
        target_class = 1,
    )
    res['name'] = name
    pipeline_results.append(res)

print('\nAll explanations complete.')

In [ ]:
# ── Step 3: side-by-side heatmaps ─────────────────────────────────────────────
for res in pipeline_results:
    print(f"\n{'═'*70}")
    print(f"  {res['name']}")
    print(f"  P(toxic) = {res['prediction_prob']:.4f}  |  "
          f"{'Toxic' if res['predicted_class']==1 else 'Non-toxic'}")
    print(f"{'═'*70}")

    visualize_explanation(res, figsize=(13, 4), atom_threshold=0.35, bond_threshold=0.35)

    mol    = Chem.MolFromSmiles(res['smiles'])
    ranked = sorted(enumerate(res['atom_importance']), key=lambda x: x[1], reverse=True)[:5]
    print('  Top-5 atoms:')
    for rank, (idx, imp) in enumerate(ranked, 1):
        sym = mol.GetAtomWithIdx(idx).GetSymbol()
        bar = '█' * int(imp * 12)
        print(f'    {rank}. Atom {idx:>3} ({sym:>2})  {bar:<14}  {imp:.4f}')

### Aggregate structural pattern — what do toxic hits share?

In [ ]:
from src.gnn_explainer import aggregate_atom_importance, plot_element_importance

for r in pipeline_results:
    r['true_label'] = 1

element_scores = aggregate_atom_importance(pipeline_results, label_filter=1)

print('Mean atom importance by element (top-3 toxic hits):')
for elem, score in sorted(element_scores.items(), key=lambda x: x[1], reverse=True):
    print(f'  {elem:<3}  {"█" * int(score * 15):<16}  {score:.4f}')

plot_element_importance(
    element_scores,
    title = 'Shared structural patterns across top-3 toxic predictions',
)

---
# Summary

## What this system produces

| Output | Scientific use |
|---|---|
| Ranked probability table | Deprioritize flagged hits before synthesis |
| Probability distribution | Assess model confidence and class separation |
| Reference panel check | Validate against established toxicity knowledge |
| Atom/bond heatmap | Pinpoint the structural feature driving toxicity |
| Ranked atom table | Guide bioisosteric replacement |
| Element aggregation | Identify shared structural alerts across a series |

## Recommended decision workflow

```
New compound library
        │
  predict_batch()         ← fast, all compounds
        │
  P(toxic) ≥ 0.5?         ← filter
  ├─ Yes → explain_molecule()  ← per flagged hit
  │            │
  │       atom heatmap + ranked table
  │            │
  │       propose modification → re-run → confirm P drops
  │
  └─ No  → advance to next testing stage
```

## Known limitations

- **Organometallics / coordination complexes**: RDKit cannot featurise Pt, Pd, Ir  
  chelates — these return `'Parse error'`. The model is trained on organic SMILES only.
- **Graph-pathway explanations only**: GNNExplainer explains GATv2, not the Transformer  
  SMILES encoder. The SMILES encoder dominates prediction (AUC 0.980 alone).  
  Treat atom/bond scores as structural hypotheses, not definitive attributions.
- **Validate against toxicophore databases**: cross-reference flagged atoms with  
  PAINS, Brenk, or FDA structural alerts before making synthesis decisions.